# 18. 텍스트 단어 연관 규칙 분석 (Association Rule Text)

`Pinocchio.txt` 대사 텍스트를 전처리하고 `TransactionEncoder`와 `apriori`, `association_rules`를 적용해 함께 등장하는 단어 패턴 및 바이그램(Bigram)을 시각화합니다.

In [ ]:
import re
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

candidates = [
    Path('Study_Python/[7] Machine_Learning/data/Pinocchio.txt'),
    Path('../[7] Machine_Learning/data/Pinocchio.txt'),
    Path('data/Pinocchio.txt'),
]
txt_path = next((p for p in candidates if p.is_file()), None)
lines = txt_path.read_text(encoding='utf-8', errors='ignore').splitlines()

stop_words = set(ENGLISH_STOP_WORDS) | {'oh', 'ah', 'uh', 'hey', 'let', 'got', 'know', 'come', 'going', 'say', 'said'}
transactions = []
for line in lines:
    words = re.findall(r"[a-z]+", line.lower())
    filtered = sorted(set(w for w in words if len(w) > 2 and w not in stop_words))
    if len(filtered) >= 2:
        transactions.append(filtered)

encoder = TransactionEncoder()
df_trans = pd.DataFrame(encoder.fit_transform(transactions), columns=encoder.columns_)

## 문제 1. 텍스트 트랜잭션에서 Apriori 연관 규칙 추출하기

`df_trans`에 대해 `min_support=5/len(df_trans)`, `max_len=2`로 빈발 항목을 찾고, `metric="confidence"`, `min_threshold=0.2`로 연관 규칙을 생성한 뒤 `lift > 1.2` 조건으로 필터링해 상위 5개 규칙을 출력하세요.

In [ ]:
frequent_items = apriori(df_trans, min_support=5 / len(df_trans), use_colnames=True, max_len=2)
rules = association_rules(frequent_items, metric='confidence', min_threshold=0.2)
rules = rules[rules['lift'] > 1.2].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5))

**결과 해석**

같은 대사 줄에 함께 등장하는 단어 쌍 중 지지도·신뢰도·향상도 기준을 넘는 핵심 연관어(예: 등장인물과 관련 호칭)를 정량적으로 도출할 수 있습니다.

## 문제 2. 인접 단어 쌍(Bigram) 상위 5개 추출하기

불용어를 제외한 단어 리스트에서 문장 내 연속으로 등장하는 2개 단어(Bigram)를 `Counter`로 집계하고 상위 5개 쌍과 빈도를 출력하세요.

In [ ]:
bigram_counts = Counter()
for line in lines:
    for part in re.split(r'[.!?,;:]+', line.lower()):
        words = [w for w in re.findall(r'[a-z]+', part) if len(w) > 2 and w not in stop_words]
        for i in range(len(words) - 1):
            bigram_counts[(words[i], words[i + 1])] += 1

print(bigram_counts.most_common(5))

**결과 해석**

동시 출현(Apriori)과 달리 바이그램(Bigram)은 실제 문맥상 바로 이어서 쓰이는 관용 표현과 호칭 순서를 포착합니다.